<h3><b>Implementation of Deep Feed Forward Neural Network (DFNN) with Adam Algorithm and Batch Normalization Technique for Multiclass Classification Tasks with Learning Rate Decay</b></h3>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time 
import h5py

In [2]:
def onehot_encoding(Y):
    
    num_classes = len(np.unique(Y))
    one_hot_Y = np.eye(num_classes)[Y].T

    return one_hot_Y

In [3]:
def layer_sizes(X, Y, num_hidden_units):
    
    num_input_units = X.shape[0]
    num_output_units = Y.shape[0]
    num_layer_units = [num_input_units] + list(num_hidden_units) + [num_output_units]

    return num_layer_units

In [4]:
def initialize_parameters(num_layer_units):
    parameters = {}
    L = len(num_layer_units)

    for l in range(1, L):
        parameters[f"W{l}"] = np.random.randn(num_layer_units[l], num_layer_units[l-1]) * np.sqrt(2 / num_layer_units[l-1])
        parameters[f"b{l}"] = np.zeros((num_layer_units[l], 1))
        
    return parameters

In [5]:
def initialize_adam_parameters(parameters):
    
    V = {}
    S = {}
    L = len(parameters) // 2
    
    for l in range(1, L+1):
        
        V[f"vdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
        V[f"vdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

        S[f"sdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
        S[f"sdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

    return (V, S)

In [6]:
def relu(Z):

    A = np.maximum(0, Z)
    activation_cache = Z

    return (A, activation_cache)

In [7]:
def backward_relu(dA, Z):

    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    
    return dZ

In [8]:
def softmax(Z):

    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True)+1e-15)
                  
    activation_cache = Z
    
    return (A, activation_cache)

In [9]:
def backward_softmax(dAL, Z):
    
    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True)+1e-15)
    dZ = A * (1 - A) * dAL  
    return dZ

In [10]:
def forward_propagation(X, parameters):

    A = X
    L = len(parameters) // 2
    caches = []
    
    for l in range(1, L):
        
        W = parameters[f"W{l}"]
        b = parameters[f"b{l}"]
        A_prev = A
        linear_cache = (A_prev, W, b)
        
        Z = np.dot(W, A_prev) + b
        A, activation_cache = relu(Z)

        caches.append((linear_cache, activation_cache))

    W =  parameters[f"W{L}"] 
    b = parameters[f"b{L}"]
    A_prev = A
    linear_cache = (A_prev, W, b)
        
    Z = np.dot(W, A_prev) + b
    A, activation_cache = softmax(Z)

    caches.append((linear_cache, activation_cache))

    return A, caches 

In [11]:
def compute_cost(A, Y):
  
    #  m = len(Y)
    A = np.clip(A, 1e-15, 1 - 1e-15)
    
    cost = - np.sum(Y * np.log(A)) #  / m

    return cost

In [12]:
def backward_propagation(AL, Y, caches):
    
    Y = Y.reshape(AL.shape)
    L = len(caches)
    gradients = {}

    dAL = - (np.divide(Y, AL + 1e-15) - np.divide(1 - Y, 1 - AL + 1e-15))

    (linear_cache, activation_cache) = caches[-1]
    (A_prev, W, b) = linear_cache
    m = A_prev.shape[1]

    dZ = backward_softmax(dAL, activation_cache)
    gradients[f"dA{L-1}"] = np.dot(W.T, dZ)
    gradients[f"dW{L}"] = 1 / m * np.dot(dZ, A_prev.T)
    gradients[f"db{L}"] = 1 / m * np.sum(dZ, axis=1, keepdims=True)

    for l in reversed(range(L - 1)):

        (linear_cache, activation_cache) = caches[l]
        (A_prev, W, b) = linear_cache
        m = A_prev.shape[1]

        dZ = backward_relu(gradients[f"dA{l+1}"], activation_cache)
        gradients[f"dA{l}"] = np.dot(W.T, dZ)
        gradients[f"dW{l+1}"] = 1 / m * np.dot(dZ, A_prev.T)
        gradients[f"db{l+1}"] = 1 / m * np.sum(dZ, axis=1, keepdims=True)

    return gradients

In [13]:
def compute_momentum(parameters, gradients, V, beta, beta_correction, t):
    
    L = len(parameters) // 2

    for l in range(1, L + 1):
        
        V[f"vdW{l}"] = (beta * V[f"vdW{l}"]) + ((1 - beta) * gradients[f"dW{l}"])
        V[f"vdb{l}"] = (beta * V[f"vdb{l}"]) + ((1 - beta) * gradients[f"db{l}"])
        
        # bias correction terms
        if beta_correction:
            V[f"vdW{l}"] = V[f"vdW{l}"] / (1 - np.power(beta, t))
            V[f"vdb{l}"] = V[f"vdb{l}"] / (1 - np.power(beta, t))
        
    return V

In [14]:
def compute_squared_gradients(parameters, gradients, S, beta, beta_correction, t):
    
    L = len(parameters) // 2

    for l in range(1, L + 1):
        
        S[f"sdW{l}"] = (beta * S[f"sdW{l}"]) + ((1 - beta) * np.power(gradients[f"dW{l}"], 2))
        S[f"sdb{l}"] = (beta * S[f"sdb{l}"]) + ((1 - beta) * np.power(gradients[f"db{l}"], 2))
        
        # bias correction terms
        if beta_correction:
            S[f"sdW{l}"] = S[f"sdW{l}"] / (1 - np.power(beta, t))
            S[f"sdb{l}"] = S[f"sdb{l}"] / (1 - np.power(beta, t))
        
    return S 

In [15]:
def update_parameters(parameters, V, S, learning_rate, epsilon):

    L = len(parameters) // 2

    for l in range(1, L + 1):
        
        parameters[f"W{l}"] -= learning_rate * (V[f"vdW{l}"] / np.sqrt(S[f"sdW{l}"] + epsilon))
        parameters[f"b{l}"] -= learning_rate * (V[f"vdb{l}"] / np.sqrt(S[f"sdb{l}"] + epsilon))

    return parameters

In [16]:
def update_learning_rate(init_learning_rate, epoch_num, decay_rate, time_interval):
  
    learning_rate = (1 * init_learning_rate) / (1 + decay_rate * (np.floor(epoch_num / time_interval)))
    
    return learning_rate

In [17]:
def predict(X, parameters):

    A, _ = forward_propagation(X, parameters)

    return A

In [18]:
def accuracy(A, Y):
    
    assert(A.shape == Y.shape)
    
    predicted_labels = np.argmax(A, axis=0)
    true_labels = np.argmax(Y, axis=0)
    
    accuracy = np.mean(predicted_labels == true_labels)

    return accuracy

In [19]:
def random_mini_batches(X, Y, mini_batch_size, seed):

    np.random.seed(seed) 
    m = X.shape[1] 
    n = Y.shape[0]
    mini_batches = []
        
    
    permutation = np.random.permutation(m)
    shuffled_X = X[:, permutation]
    shuffled_Y = Y[:, permutation].reshape((n, m))
    
    # Step 2: Partition the shuffled data into mini-batches
    num_complete_minibatches = m // mini_batch_size  
    
    for k in range(num_complete_minibatches):
        start_idx = k * mini_batch_size
        end_idx = (k + 1) * mini_batch_size
        mini_batch_X = shuffled_X[:, start_idx:end_idx]
        mini_batch_Y = shuffled_Y[:, start_idx:end_idx]
        mini_batches.append((mini_batch_X, mini_batch_Y))
    
    # Handling the end case (last mini-batch may have fewer examples)
    if m % mini_batch_size != 0:
        start_idx = num_complete_minibatches * mini_batch_size
        mini_batch_X = shuffled_X[:, start_idx:]
        mini_batch_Y = shuffled_Y[:, start_idx:]
        mini_batches.append((mini_batch_X, mini_batch_Y))
    
    return mini_batches

In [20]:
def train(X_train, Y_train, X_test, Y_test, num_hidden_units, num_epochs=1000, learning_rate=1e-3, beta1=0.9, beta2=0.999,
          decay_rate=0.3, time_interval=100, mini_batch_size=64, beta_correction=False, epsilon=1e-8, print_cost=100, seed=1):
    
    train_time = 0
    cost_ = []
    learning_rates = [learning_rate]
   
    Y_train = onehot_encoding(Y_train)
    Y_test = onehot_encoding(Y_test)
    
    num_layer_units = layer_sizes(X_train, Y_train, num_hidden_units)
    parameters = initialize_parameters(num_layer_units)
    (V, S) = initialize_adam_parameters(parameters)
    
    init_learning_rate = learning_rate
    epoch_num = 0
    m = X_train.shape[1]
    t = 0
    for i in range(num_epochs):

        tic = time.time()
        seed = seed + 1
        minibatches = random_mini_batches(X_train, Y_train, mini_batch_size, seed)
        cost_total = 0
        
        for minibatch in minibatches:

            (X_minibatch, Y_minibatch) = minibatch 
            A_minibatch, minibatch_caches = forward_propagation(X_minibatch, parameters)
            cost_total += compute_cost(A_minibatch, Y_minibatch)
            gradients = backward_propagation(A_minibatch, Y_minibatch, minibatch_caches)
            t += 1
            V = compute_momentum(parameters, gradients, V, beta1, beta_correction, t)
            S = compute_squared_gradients(parameters, gradients, S, beta2, beta_correction, t)
            parameters = update_parameters(parameters, V, S, learning_rate, epsilon)
        
        toc = time.time()
        train_time += toc - tic
        cost_.append(cost_total / m)
        epoch_num += 1
        learning_rate = update_learning_rate(init_learning_rate, epoch_num, decay_rate, time_interval)
        learning_rates.append(learning_rate)
        if i % print_cost == 0:
            print(f"Epoch: {i}; Cost: {cost_total / m}")
            train_accuracy = accuracy(predict(X_train, parameters), Y_train)
            test_accuracy = accuracy(predict(X_test, parameters), Y_test)
            print(f"Train Accuracy: {train_accuracy}")
            print(f"Test Accuracy: {test_accuracy}")
            print()
            
    train_accuracy = accuracy(predict(X_train, parameters), Y_train)
    test_accuracy = accuracy(predict(X_test, parameters), Y_test)
    
    print(f"Train Accuracy: {train_accuracy}")
    print(f"Test Accuracy: {test_accuracy}")
    print(f"Train Duration: {train_time}")

    plt.plot(cost_)
    plt.title("epoch vs. cost")
    plt.xlabel("# epoch")
    plt.ylabel("cost")
    
    return (parameters, cost_, train_time, learning_rates) 